# Length generalization: original FoX versus our model setup

Train the original plain FoX (LLaMA) recipe alongside our optimizers and first-gate interventions, using equal natural-text token budgets. The full matrix has **10 branches**:

- Original data-dependent FoX: paper AdamW, our fixed Adam, annealed Adam, and SGD.
- Our first gate `g=softplus(u*v)`: our fixed Adam, annealed Adam, and SGD.
- Direct first-gate control `g=softplus(z)`: the same three optimizers.

All branches share the initial **non-gate backbone weights** and input batches. Factorized/direct constant pairs start with matching functions and positive balanced factors; their initial gates differ from original FoX. Other architectural defaults are held at the paper backbone so the comparison focuses on our gate and optimizer setup.

**Start with `PROFILE="smoke"`; switch to `"colab"` with a GPU for training.** Read [the design and metric definitions](../docs/length_generalization_comparison.md). These finite language-model tests do not establish infinite retrieval or the controlled model's certificate.

## 1. Open the repository

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = ""  # After publishing: https://github.com/YOUR_NAME/fox_experiments.git
REPO_REF = "main"  # Branch, tag, or commit to use in a new Colab clone.
candidates = [Path.cwd(), Path.cwd().parent, Path("/content/fox_experiments")]
REPO = next((p for p in candidates if (p / "src/fox_experiments").is_dir()), None)
if REPO is None:
    if not REPO_URL:
        raise ValueError("Set REPO_URL above, or open this notebook inside a local clone.")
    REPO = Path("/content/fox_experiments")
    subprocess.check_call(["git", "clone", REPO_URL, str(REPO)])
    subprocess.check_call(["git", "-C", str(REPO), "checkout", REPO_REF])
REPO = REPO.resolve()
os.chdir(REPO)
print("Repository:", REPO)
print(subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()
      if (REPO / ".git/HEAD").exists() and subprocess.run(
          ["git", "rev-parse", "--verify", "HEAD"], capture_output=True).returncode == 0
      else "No commit yet: commit the source before a scientific run.")

## 2. Install and check the runtime

In [ ]:
if os.environ.get("FOX_SKIP_INSTALL") != "1":
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO)])
sys.path.insert(0, str(REPO / "src"))  # Make this checkout visible to the current kernel.

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cpu":
    torch.set_num_threads(min(4, os.cpu_count() or 1))
print("PyTorch:", torch.__version__, "| Device:", DEVICE)

## 3. Choose the budget, variants, and length grid

The Colab preset trains on 256-token windows and evaluates 128, 256, 512, 1,024 and 2,048-token windows. Every branch predicts identical held-out target suffixes. Ten branches at 4,096,000 tokens each require 40,960,000 training tokens in total.

Edit `configs/length_generalization/colab.json` for training steps, seeds, rates, or the length grid. Rates are predeclared starting settings, not optima. There is no task-acquisition stage or optimizer reset during a branch. Use a fresh run name after any change. Set `RESUME=True` only for an identical interrupted run.

In [ ]:
from fox_experiments.paper_baseline import PaperBaselineConfig
from IPython.display import display, Image
import pandas as pd

PROFILE = os.environ.get("FOX_LENGTH_PROFILE", "smoke")  # smoke / colab
RUN_NAME = os.environ.get("FOX_LENGTH_RUN_NAME", PROFILE + "_v1")
USE_GOOGLE_DRIVE = False
RESUME = False
RUN_TRAINING = True
WORKSPACE = Path(os.environ.get("FOX_LENGTH_WORKSPACE", str(REPO / "outputs/length_generalization")))
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORKSPACE = Path("/content/drive/MyDrive/fox_experiments/length_generalization")
OUT = WORKSPACE / RUN_NAME
DATA_DIR = Path(os.environ.get("FOX_DATA_DIR", str(REPO / "data/longcrawl64_pilot")))
if USE_GOOGLE_DRIVE:
    DATA_DIR = WORKSPACE / "data/longcrawl64_pilot"
CONFIG = PaperBaselineConfig.from_json(REPO / "configs/length_generalization" / (PROFILE + ".json"))
print("Output:", OUT)
from fox_experiments.paper_baseline.config import branch_specs
print("Tokens per branch:", CONFIG.steps * CONFIG.tokens_per_update)
display(pd.DataFrame(branch_specs(CONFIG)))
display(pd.Series(CONFIG.to_dict(), name="Setting").to_frame())
if RUN_TRAINING and CONFIG.profile != "smoke" and DEVICE != "cuda":
    raise RuntimeError("Choose a GPU runtime for the Colab pilot, or PROFILE='smoke'.")

## 4. Prepare the native text subset

A fresh clone downloads the verified pilot corpus once. Training uses text only, with a BOS/EOT input reset and equal token weights. Pilot documents repurpose disjoint native heldout rows; the published full run instead uses native `train.zarr`.

In [ ]:
from fox_experiments.cli import prepare_data

_, manifest = prepare_data(DATA_DIR)
print("Corpus SHA256:", manifest["sha256"])
print("Split counts:", manifest["split_counts"])

## 5. Train the matched comparison

Implementation is in `src/fox_experiments/paper_baseline/`. `build_comparison_fox` applies our gate initialization and factorization to the shared paper backbone; the runner records separate branch and shared-backbone hashes.

The original branch uses the paper AdamW recipe. Our branches use the same optimizer settings as the existing experiment: configured moments/epsilon schedules, polynomial rates, and gate/output multipliers. Each branch keeps its own optimizer state throughout training.

Smoke verifies the software only. A branch changing architecture or forgetting initialization is not an optimizer-only comparison.

In [ ]:
from fox_experiments.paper_baseline import run_paper_comparison

if RUN_TRAINING:
    artifacts = run_paper_comparison(CONFIG, DATA_DIR, OUT, device=DEVICE, resume=RESUME)
    print(artifacts)
else:
    print("Analyze the existing run at:", OUT)

## 6. Read the length-generalization comparison

For each branch and context length, the report includes:

- **NLL:** lower is better, evaluated on the same targets.
- **Added-context gain:** NLL at training-length context minus NLL at this context; positive means the extra prefix helps. Lengths below the training length are controls, not extrapolation.
- **NLL difference versus paper FoX:** our NLL minus paper NLL; negative favors our branch.
- **Extra-context-gain difference versus paper FoX:** our gain minus paper gain; positive means a larger improvement from added context.

Read absolute NLL and gain together: a weak short-context model can have more room to improve. Missing/failed baselines and unmatched targets are reported explicitly. One seed gives a pilot only; repeat predeclared seeds before making a robust comparison. No long-test results select rates.

In [ ]:
from fox_experiments.paper_baseline import summarize_paper_comparison

report = summarize_paper_comparison(OUT)
print(report)
import json
print("Run status:", json.loads((OUT / "run_status.json").read_text()))
print("Failed arms:", json.loads((OUT / "failures.json").read_text()))
for filename in ("comparison_vs_paper.csv", "context_summary.csv", "summary.csv"):
    if (OUT / filename).exists():
        display(pd.read_csv(OUT / filename))
for figure in sorted(OUT.glob("*.png")):
    display(Image(filename=str(figure)))

## 7. Export the comparison

The report ZIP excludes weights and the corpus. Keep the run folder for resume or later checkpoint evaluation. The exact published-scale baseline remains a separate upstream training configuration; this notebook runs a downscaled architecture/recipe comparison.

In [ ]:
from fox_experiments.notebook_utils import archive_results

archive = archive_results(WORKSPACE / (RUN_NAME + "_results.zip"), {"paper_baseline": OUT})
print("Saved:", archive)